# ALTO v1: Adaptive Latent Optimization for Context-aware Human Generation

Pipeline: **Open Images V7 + structured prompts → semantic scene retrieval → Qwen3-VL compatibility selection → Background Pool (3–4 backgrounds per prompt) → prompt + background → box mask → SDXL Inpainting → 4 candidates → automatic evaluator → accept / stop / expand to 8 / expand to 16 → best output**.

## 1. Drive layout

```text
/content/drive/MyDrive/Colab Notebooks/alto/v1/
├── Input/
│   ├── backgrounds/
│   └── masks/
└── Output/
    ├── images/
    ├── runs.csv
    ├── evaluations.csv
    └── experiment.json
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Dependencies

Pin một dependency set tương thích cho Qwen3-VL, SDXL và Colab Gradio. Transformers 4.x sử dụng Hugging Face Hub 0.x; sau khi thay đổi versions, bắt buộc restart runtime trước khi chạy các cells còn lại.

In [ ]:
%pip uninstall -y gradio
%pip install -q --upgrade \
    datasets accelerate safetensors \
    "transformers==4.57.6" \
    "diffusers==0.35.2" \
    "requests==2.32.4"

In [ ]:
import gc
import json
import math
import os
import re
import time
from datetime import datetime
from importlib.metadata import version as package_version
from io import BytesIO
from pathlib import Path
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

HF_TOKEN = HF_TOKEN or False
if not HF_TOKEN:
    os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
import diffusers
import transformers
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from IPython.display import display
from datasets import load_dataset
from diffusers import AutoPipelineForInpainting
from transformers import (
    AutoImageProcessor,
    AutoProcessor,
    CLIPModel,
    CLIPProcessor,
    Mask2FormerForUniversalSegmentation,
    Qwen3VLForConditionalGeneration,
)

sdxl_pipe = None
instance_processor = instance_model = None
clip_processor = clip_model = None

print("Hugging Face authentication:", "enabled" if HF_TOKEN else "anonymous")
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)
print("Hugging Face Hub:", package_version("huggingface-hub"))
print("Gradio:", package_version("gradio"))
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not enabled")

## 3. Experiment configuration

Khai báo paths, execution flags và generation protocol. `full_fixed` sinh đủ 16 seeds để so sánh offline; `adaptive` dừng theo policy.


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/alto/v1")
INPUT_DIR = PROJECT_DIR / "Input"
OUTPUT_DIR = PROJECT_DIR / "Output"
BACKGROUND_DIR = INPUT_DIR / "backgrounds"
MASK_DIR = INPUT_DIR / "masks"
RUNS_PATH = OUTPUT_DIR / "runs.csv"
EVALUATIONS_PATH = OUTPUT_DIR / "evaluations.csv"
EXPERIMENT_PATH = OUTPUT_DIR / "experiment.json"

for path in [INPUT_DIR, OUTPUT_DIR, BACKGROUND_DIR, MASK_DIR, OUTPUT_DIR / "images"]:
    path.mkdir(parents=True, exist_ok=True)

# First execution: True to prepare and save a fixed Background Pool.
# Later experiment runs: set this to False to reuse exactly the same pool.
RUN_POOL_SELECTION = True
RUN_GENERATION = True
RUN_EVALUATION = True

# "full_fixed": generate all 16 seeds, then simulate ALTO offline.
# "adaptive": stop generation immediately when ALTO accepts or rejects.
GENERATION_PROTOCOL = "full_fixed"


### 3.1 Context specifications

Mỗi task khai báo `context_type`, context constraints và background requirements. Các cells lần lượt định nghĩa schema/rendering, nhóm prompts `p001–p003`, rồi `p004–p006`.

In [ ]:
CONTEXT_TYPES = {
    "independent_placement",
    "relative_placement",
    "physical_interaction",
    "scene_aware_placement",
    "orientation",
    "activity",
}


def render_generation_prompt(context):
    """Render an SDXL prompt from a structured context specification."""
    subject = context.get("subject", "adult person")
    appearance = context.get("appearance", "")
    pose = context.get("pose", "standing naturally")
    relation = context.get("relation")
    reference = context.get("reference_object")
    activity = context.get("activity")
    orientation = context.get("orientation")
    scene_hint = context.get("scene_hint", "the scene")

    parts = [f"A photorealistic full-body {subject}"]

    if appearance:
        parts.append(appearance)

    if activity:
        parts.append(activity)
    else:
        parts.append(pose)

    if relation and reference:
        parts.append(f"{relation.replace('_', ' ')} the designated {reference}")
    elif orientation and reference:
        parts.append(f"{orientation.replace('_', ' ')} the designated {reference}")

    parts.append(f"in {scene_hint}")
    core = " ".join(parts)

    return (
        f"{core}, with correct anatomy, perspective, scale, lighting, "
        "ground contact, and a natural contact shadow. Preserve the existing scene."
    )


def make_context_spec(
    prompt_id,
    context_type,
    context,
    environment,
    setting,
    contextual_elements,
    spatial_constraints,
):
    """Build and validate one context specification."""
    if context_type not in CONTEXT_TYPES:
        raise ValueError(f"Unsupported context_type: {context_type}")

    specification = {
        "prompt_id": prompt_id,
        "context_type": context_type,
        "context": context,
        "background_requirements": {
            "environment": environment,
            "setting": setting,
            "contextual_elements": list(contextual_elements),
            "spatial_constraints": list(spatial_constraints),
        },
    }
    specification["generation_prompt"] = render_generation_prompt(context)
    return specification


In [ ]:
PROMPTS = [
    # 1. Independent placement: no explicit reference object.
    make_context_spec(
        prompt_id="p001",
        context_type="independent_placement",
        context={
            "subject": "adult person",
            "pose": "standing naturally",
            "scene_hint": "a spacious indoor public area",
        },
        environment="indoor",
        setting="spacious lobby, hall, terminal, or public interior",
        contextual_elements=["visible floor", "clear environmental context"],
        spatial_constraints=[
            "unoccupied full-body standing area",
            "the target region must not cover important furniture or objects",
        ],
    ),

    # 2. Relative placement: geometric relation to an existing object.
    make_context_spec(
        prompt_id="p002",
        context_type="relative_placement",
        context={
            "subject": "adult person",
            "pose": "standing naturally",
            "relation": "next_to",
            "reference_object": "parked bicycle",
            "scene_hint": "an outdoor public space",
        },
        environment="outdoor",
        setting="street, plaza, park path, or bicycle parking area",
        contextual_elements=["one clearly visible parked bicycle", "walkable ground"],
        spatial_constraints=[
            "free standing space immediately beside the bicycle",
            "the full bicycle and the person's body should remain visible",
        ],
    ),

    # 3. Physical interaction: body contact with a reference object.
    make_context_spec(
        prompt_id="p003",
        context_type="physical_interaction",
        context={
            "subject": "adult person",
            "pose": "sitting naturally",
            "relation": "sitting_on",
            "reference_object": "chair or bench",
            "scene_hint": "an indoor or sheltered public area",
        },
        environment="indoor",
        setting="waiting area, lounge, cafe, lobby, or public seating area",
        contextual_elements=["one usable chair or bench", "visible floor around the seat"],
        spatial_constraints=[
            "the seat must be clearly visible and large enough for a person",
            "there must be legroom in front of the seat",
            "the seat must not be blocked by a table or another object",
        ],
    ),

    # 4. Scene-aware placement: relation to a stable scene landmark.
    make_context_spec(
        prompt_id="p004",
        context_type="scene_aware_placement",
        context={
            "subject": "adult person",
            "pose": "standing naturally",
            "relation": "in_front_of",
            "reference_object": "building entrance",
            "scene_hint": "an outdoor architectural scene",
        },
        environment="outdoor",
        setting="building exterior, courtyard, plaza, or entrance area",
        contextual_elements=["clearly visible building entrance", "walkable foreground"],
        spatial_constraints=[
            "free standing space in front of the entrance",
            "the person must not occlude the entire entrance",
        ],
    ),

    # 5. Orientation: body or gaze direction toward a reference.
    make_context_spec(
        prompt_id="p005",
        context_type="orientation",
        context={
            "subject": "adult person",
            "pose": "standing naturally",
            "orientation": "facing_toward",
            "reference_object": "shop display",
            "scene_hint": "an indoor retail environment",
        },
        environment="indoor",
        setting="shopping mall, store, gallery, or retail interior",
        contextual_elements=["clearly visible shop display or storefront", "open floor"],
        spatial_constraints=[
            "free standing space with a clear line of sight to the display",
            "the target box should allow the body to face the reference naturally",
        ],
    ),

    # 6. Activity: context-consistent motion rather than a static pose.
    make_context_spec(
        prompt_id="p006",
        context_type="activity",
        context={
            "subject": "adult person",
            "activity": "walking naturally along the path",
            "scene_hint": "an outdoor pedestrian environment",
        },
        environment="outdoor",
        setting="sidewalk, park path, trail, or pedestrian walkway",
        contextual_elements=["continuous visible walking path", "natural outdoor context"],
        spatial_constraints=[
            "enough longitudinal space for a walking pose",
            "the target region must align with the direction of the path",
            "feet should contact the walkable surface",
        ],
    ),
]

### 3.2 Pipeline parameters

Cell tiếp theo gom toàn bộ parameters của Background Pool, Qwen3-VL, SDXL, evaluator và adaptive budget, sau đó kiểm tra invariants.

In [ ]:
TARGET_BACKGROUND_COUNT = 4
MIN_BACKGROUND_COUNT = 3
QWEN_CANDIDATES_PER_REQUEST = 8
QWEN_REQUEST_RETRIES = 8

NEGATIVE_PROMPT = (
    "multiple people, duplicate person, cropped body, extra limbs, missing limbs, "
    "deformed hands, distorted face, floating feet, text, watermark, background distortion"
)

CONFIG = {
    "experiment_name": "ALTO v1: Adaptive Latent Optimization for Context-aware Human Generation",
    "generation_protocol": GENERATION_PROTOCOL,
    "open_images_dataset_id": "bitmind/open-images-v7",
    "open_images_reference_url": "https://storage.googleapis.com/openimages/web/download_v7.html",
    "open_images_split": "validation",
    "dataset_seed": 42,
    "target_background_count": TARGET_BACKGROUND_COUNT,
    "min_background_count": MIN_BACKGROUND_COUNT,
    "selection_candidate_count": 384,
    "selection_max_attempts": 4000,
    "request_timeout_seconds": 20,
    "min_background_width": 640,
    "min_background_height": 480,
    "candidate_preview_side": 512,
    "max_background_side": 1600,
    "qwen_model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "qwen_max_new_tokens": 1024,
    "qwen_candidates_per_request": QWEN_CANDIDATES_PER_REQUEST,
    "qwen_request_retries": QWEN_REQUEST_RETRIES,
    "sdxl_model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    "max_generation_width": 1024,
    "max_generation_height": 1024,
    "num_inference_steps": 40,
    "guidance_scale": 8.0,
    "strength": 0.99,
    "mask_blur_radius": 12,
    "seed_start": 1000,
    "round_seed_counts": [4, 4, 8],
    "instance_model_id": "facebook/mask2former-swin-small-coco-instance",
    "clip_model_id": "openai/clip-vit-base-patch32",
    "retrieval_batch_size": 16,
    "person_detection_threshold": 0.35,
    "min_placement": 0.35,
    "min_scale": 0.25,
    "min_prompt_alignment": 0.55,
    "min_background_preservation": 0.95,
    "promising_margin": 0.05,
}

assert CONFIG["generation_protocol"] in {"full_fixed", "adaptive"}
assert len({item["prompt_id"] for item in PROMPTS}) == len(PROMPTS)
assert 1 <= CONFIG["min_background_count"] <= CONFIG["target_background_count"]
assert sum(CONFIG["round_seed_counts"]) == 16
assert CONFIG["selection_candidate_count"] >= CONFIG["target_background_count"]
assert CONFIG["selection_candidate_count"] >= (
    CONFIG["qwen_candidates_per_request"] * CONFIG["qwen_request_retries"]
)
print(json.dumps(CONFIG, indent=2))

## 4. Open Images V7 candidate pool

Cell này định nghĩa image resizing, HTTP download và deterministic candidate collection ở preview resolution. CLIP ranking và Qwen selection được xử lý ở section kế tiếp.

In [ ]:
def resize_long_side(image, max_side):
    """Resize an image while preserving its aspect ratio."""
    if max(image.size) <= max_side:
        return image.copy()
    scale = max_side / max(image.size)
    size = tuple(max(1, round(value * scale)) for value in image.size)
    return image.resize(size, Image.Resampling.LANCZOS)


def download_open_image(url, max_side):
    """Download, validate, orient and resize one Open Images sample."""
    response = requests.get(
        url,
        timeout=CONFIG["request_timeout_seconds"],
        headers={"User-Agent": "ALTO research notebook"},
    )
    response.raise_for_status()
    image = ImageOps.exif_transpose(Image.open(BytesIO(response.content))).convert("RGB")
    if (
        image.width < CONFIG["min_background_width"]
        or image.height < CONFIG["min_background_height"]
    ):
        raise ValueError(f"Image quá nhỏ: {image.size}")
    return resize_long_side(image, max_side)


def collect_background_candidates():
    """Collect a deterministic preview pool from the validation stream."""
    stream = load_dataset(
        CONFIG["open_images_dataset_id"],
        split=CONFIG["open_images_split"],
        streaming=True,
        token=HF_TOKEN,
    ).shuffle(seed=CONFIG["dataset_seed"], buffer_size=500)
    candidates = []
    skip_counts = {}
    for attempt, row in enumerate(stream):
        if attempt >= CONFIG["selection_max_attempts"]:
            break
        try:
            image = download_open_image(
                row["url"], CONFIG["candidate_preview_side"]
            )
        except Exception as exc:
            error_type = type(exc).__name__
            skip_counts[error_type] = skip_counts.get(error_type, 0) + 1
            continue
        source_index = row.get("index")
        candidates.append({
            "pool_index": len(candidates),
            "source_index": attempt if source_index is None else int(source_index),
            "source_url": row["url"],
            "image": image,
        })
        if len(candidates) >= CONFIG["selection_candidate_count"]:
            break
    if len(candidates) < CONFIG["selection_candidate_count"]:
        raise RuntimeError(
            f"Chỉ tải được {len(candidates)} Open Images candidates; "
            f"skipped={skip_counts}"
        )
    print(f"Collected {len(candidates)} candidates; skipped={skip_counts}")
    return candidates


if RUN_POOL_SELECTION:
    background_candidates = collect_background_candidates()

## 5. Qwen3-VL Background Pool

Qwen3-VL đánh giá scene compatibility và chọn 3–4 backgrounds cho mỗi prompt. Các cells lần lượt xử lý response utilities, CLIP ranking, schema validation, multimodal prompt, inference, retries và pool orchestration.

In [ ]:
def parse_json_object(text):
    """Extract the first valid JSON object from model output."""
    decoder = json.JSONDecoder()
    for start in (match.start() for match in re.finditer(r"\{", text)):
        try:
            value, _ = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    raise ValueError(f"Qwen response không chứa JSON object hợp lệ: {text}")


def pooled_feature_tensor(model_output):
    """Normalize CLIP feature return types across Transformers versions."""
    if torch.is_tensor(model_output):
        return model_output

    pooled_output = getattr(model_output, "pooler_output", None)
    if torch.is_tensor(pooled_output):
        return pooled_output

    raise TypeError(
        "Feature extractor phải trả về Tensor hoặc model output có pooler_output"
    )


def build_background_query(prompt_spec):
    """Convert background requirements into a CLIP retrieval query."""
    requirements = prompt_spec["background_requirements"]
    elements = "; ".join(requirements["contextual_elements"])
    return (
        f"A wide-angle environmental photograph of an {requirements['environment']} "
        f"{requirements['setting']}, showing {elements}, visible ground, natural perspective, "
        "and open space where a full-body person could stand."
    )


In [ ]:
@torch.inference_mode()
def build_semantic_rankings(candidates, prompts):
    """Rank candidate backgrounds for every prompt with CLIP."""
    assert torch.cuda.is_available(), "Semantic retrieval cần GPU"
    retrieval_processor = CLIPProcessor.from_pretrained(
        CONFIG["clip_model_id"], token=HF_TOKEN
    )
    retrieval_model = CLIPModel.from_pretrained(
        CONFIG["clip_model_id"], token=HF_TOKEN
    ).to("cuda").eval()

    image_features = []
    batch_size = CONFIG["retrieval_batch_size"]
    for start in range(0, len(candidates), batch_size):
        images = [item["image"] for item in candidates[start:start + batch_size]]
        inputs = retrieval_processor(images=images, return_tensors="pt").to("cuda")
        features = F.normalize(
            pooled_feature_tensor(
                retrieval_model.get_image_features(pixel_values=inputs.pixel_values)
            ),
            dim=-1,
        )
        image_features.append(features.cpu())

    image_features = torch.cat(image_features, dim=0)
    negative_query = (
        "A close-up, macro, portrait, isolated product, food, drink, small object, animal, "
        "bird, aircraft, or mostly sky photograph without a walkable environmental scene."
    )
    queries = [build_background_query(item) for item in prompts] + [negative_query]
    text_inputs = retrieval_processor(
        text=queries,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to("cuda")
    text_features = F.normalize(
        pooled_feature_tensor(
            retrieval_model.get_text_features(
                input_ids=text_inputs.input_ids,
                attention_mask=text_inputs.attention_mask,
            )
        ),
        dim=-1,
    ).cpu()

    shortlist_size = (
        CONFIG["qwen_candidates_per_request"] * CONFIG["qwen_request_retries"]
    )
    rankings = {}
    for prompt_index, prompt_spec in enumerate(prompts):
        scores = (
            image_features @ text_features[prompt_index]
            - image_features @ text_features[-1]
        )
        order = torch.argsort(scores, descending=True)[:shortlist_size].tolist()
        rankings[prompt_spec["prompt_id"]] = [
            {**candidates[index], "retrieval_score": float(scores[index].item())}
            for index in order
        ]
        print(
            prompt_spec["prompt_id"],
            "semantic shortlist",
            [candidates[index]["pool_index"] for index in order[:8]],
        )

    del retrieval_model, retrieval_processor
    del image_features, text_features, inputs, text_inputs
    gc.collect()
    torch.cuda.empty_cache()
    return rankings


In [ ]:
def validate_background_selection(response, valid_image_indices, prompt_spec, max_items):
    """Validate Qwen selections, normalized boxes and candidate indices."""
    selections = response.get("backgrounds", response.get("selected_backgrounds", []))
    if not isinstance(selections, list):
        raise ValueError("backgrounds phải là một list")
    if len(selections) > max_items:
        raise ValueError(
            f"{prompt_spec['prompt_id']}: tối đa {max_items} backgrounds, "
            f"nhưng nhận được {len(selections)}"
        )

    validated = []
    batch_indices = set()
    expected_environment = prompt_spec["background_requirements"]["environment"]
    conflicting_environment = "outdoor" if expected_environment == "indoor" else "indoor"

    for selection in selections:
        image_index = int(selection["image_index"])
        box_values = selection.get("box_2d", selection.get("target_box"))
        if box_values is None:
            raise ValueError("Selection thiếu box_2d")

        box = [float(value) for value in box_values]
        scene_type = str(
            selection.get("scene_type", expected_environment)
        ).lower().strip()
        matched_elements = selection.get("matched_elements", [])
        if isinstance(matched_elements, str):
            matched_elements = [matched_elements]
        matched_elements = [str(value) for value in matched_elements]
        match_reason = str(selection.get("match_reason", "not provided")).strip()

        if image_index not in valid_image_indices:
            raise ValueError(
                f"image_index không thuộc candidate window: {image_index}"
            )
        if image_index in batch_indices:
            raise ValueError(
                f"image_index bị lặp trong cùng response: {image_index}"
            )
        if (
            conflicting_environment in scene_type
            and expected_environment not in scene_type
        ):
            raise ValueError(
                f"Expected {expected_environment}, received {scene_type}"
            )
        if len(box) != 4:
            raise ValueError("box_2d phải có bốn coordinates")

        x1, y1, x2, y2 = box
        if not (0 <= x1 < x2 <= 1000 and 0 <= y1 < y2 <= 1000):
            raise ValueError(f"box_2d ngoài miền 0..1000: {box}")
        if x2 - x1 < 100 or y2 - y1 < 250:
            raise ValueError(f"box_2d quá nhỏ cho full-body person: {box}")

        batch_indices.add(image_index)
        validated.append({
            "prompt_id": prompt_spec["prompt_id"],
            "image_index": image_index,
            "box_2d": box,
            "scene_type": expected_environment,
            "matched_elements": matched_elements,
            "match_reason": match_reason,
        })

    return validated


In [ ]:
def build_selection_content(candidates, prompt_spec, max_items):
    """Build the multimodal Qwen request for one candidate window."""
    if not candidates:
        raise ValueError("Candidate window rỗng")

    content = []
    for item in candidates:
        content.extend([
            {"type": "text", "text": f"IMAGE_INDEX={item['pool_index']}"},
            {"type": "image", "image": resize_long_side(item["image"], 512)},
        ])

    instruction = f"""
Select 1 to {max_items} suitable backgrounds.
Select at least one whenever any candidate satisfies all hard constraints.
Return an empty list only if every candidate fails a hard constraint.

GENERATION_PROMPT={prompt_spec['generation_prompt']}
BACKGROUND_REQUIREMENTS={json.dumps(prompt_spec['background_requirements'], ensure_ascii=False)}

Selection priorities, in order:
1. Hard: correct environment, visible ground, free full-body space, and no visible person.
2. The setting must be semantically compatible, but does not need to match the wording exactly.
3. Prefer at least one contextual element; never claim an absent element.
4. Reject close-ups, isolated objects, mostly-sky images, or conflicting scenes.
5. Place the target box on free space without covering important contextual objects.
6. Prefer diversity in viewpoint and composition among otherwise compatible images.

Rules:
- Coordinates use [x1, y1, x2, y2] normalized to integers from 0 to 1000.
- Use only the visible IMAGE_INDEX labels.
- Never repeat an image_index.
- scene_type must be exactly "{prompt_spec['background_requirements']['environment']}".
- Return JSON only.
- Every selected object must contain image_index, box_2d, scene_type,
  matched_elements, and match_reason.
- A valid empty response is: {{"backgrounds": []}}

Valid structure:
{{"backgrounds": [
  {{"image_index": {candidates[0]['pool_index']}, "box_2d": [350, 100, 650, 900],
    "scene_type": "{prompt_spec['background_requirements']['environment']}",
    "matched_elements": ["visible element"],
    "match_reason": "short reason"}}
]}}
"""
    content.append({"type": "text", "text": instruction})
    return content


In [ ]:
@torch.inference_mode()
def select_prompt_backgrounds(
    model, processor, candidates, prompt_spec, max_items
):
    """Run one Qwen selection request and validate its response."""
    content = build_selection_content(candidates, prompt_spec, max_items)
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    inputs = inputs.to(model.device)

    generated = model.generate(
        **inputs,
        max_new_tokens=CONFIG["qwen_max_new_tokens"],
        do_sample=False,
    )
    trimmed = [
        output[len(source):]
        for source, output in zip(inputs.input_ids, generated)
    ]
    response = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    selections = validate_background_selection(
        parse_json_object(response),
        {item["pool_index"] for item in candidates},
        prompt_spec,
        max_items=max_items,
    )
    del inputs, generated
    return selections, response


In [ ]:
def load_qwen_selector():
    """Load Qwen3-VL-4B and its processor for background selection."""
    assert torch.cuda.is_available(), "Qwen3-VL selection cần GPU"
    processor = AutoProcessor.from_pretrained(
        CONFIG["qwen_model_id"],
        token=HF_TOKEN,
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG["qwen_model_id"],
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
        token=HF_TOKEN,
    ).eval()
    return model, processor

In [ ]:
def select_ranked_candidates(model, processor, prompt_spec, ranked_candidates):
    """Retry disjoint ranking windows until the prompt quota is met."""
    prompt_id = prompt_spec["prompt_id"]
    selections = []
    selected_indices = set()
    attempt_log = []
    last_error = None
    window_size = CONFIG["qwen_candidates_per_request"]

    for attempt in range(CONFIG["qwen_request_retries"]):
        remaining = CONFIG["target_background_count"] - len(selections)
        if remaining <= 0:
            break

        start = attempt * window_size
        window = ranked_candidates[start:start + window_size]
        if not window:
            break
        window_indices = [item["pool_index"] for item in window]

        try:
            batch, response = select_prompt_backgrounds(
                model, processor, window, prompt_spec, max_items=remaining
            )
            batch = [
                item for item in batch
                if item["image_index"] not in selected_indices
            ]
            selections.extend(batch)
            selected_indices.update(item["image_index"] for item in batch)
            attempt_log.append({
                "prompt_id": prompt_id,
                "attempt": attempt + 1,
                "status": "success" if batch else "empty",
                "candidate_window": window_indices,
                "selected_indices": [item["image_index"] for item in batch],
                "response": response,
            })
            print(
                prompt_id, f"window {attempt + 1}:",
                [item["image_index"] for item in batch],
                f"total={len(selections)}",
            )
        except Exception as exc:
            last_error = exc
            attempt_log.append({
                "prompt_id": prompt_id,
                "attempt": attempt + 1,
                "status": "failed",
                "candidate_window": window_indices,
                "error": repr(exc),
            })
            print(f"Retry {prompt_id} ({attempt + 1}): {exc}")
            gc.collect()
            torch.cuda.empty_cache()

    minimum = CONFIG["min_background_count"]
    if len(selections) < minimum:
        status_counts = {}
        for item in attempt_log:
            status = item["status"]
            status_counts[status] = status_counts.get(status, 0) + 1
        raise RuntimeError(
            f"{prompt_id}: chỉ chọn được {len(selections)} backgrounds, "
            f"yêu cầu tối thiểu {minimum}; attempts={status_counts}"
        ) from last_error

    return selections[:CONFIG["target_background_count"]], attempt_log

In [ ]:
def build_background_pool(candidates, prompts):
    """Build selections for all prompts and release Qwen resources safely."""
    rankings = build_semantic_rankings(candidates, prompts)
    model, processor = load_qwen_selector()
    all_selections = []
    attempt_log = []

    try:
        for prompt_spec in prompts:
            prompt_id = prompt_spec["prompt_id"]
            selections, prompt_log = select_ranked_candidates(
                model, processor, prompt_spec, rankings[prompt_id]
            )
            all_selections.extend(selections)
            attempt_log.extend(prompt_log)
    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

    return all_selections, attempt_log


if RUN_POOL_SELECTION:
    qwen_selections, qwen_selection_log = build_background_pool(
        background_candidates, PROMPTS
    )
    print(qwen_selections)

## 6. Simple box-based mask

Cell utilities chuyển Qwen box sang pixel coordinates và tạo binary mask. Cell materialization tải ảnh đã chọn, lưu Background Pool và cập nhật `experiment.json`.

In [ ]:
def normalized_box_to_pixels(box_2d, image_size):
    """Convert a Qwen box from 0..1000 coordinates to image pixels."""
    width, height = image_size
    x1, y1, x2, y2 = box_2d
    box = (
        max(0, min(width - 1, round(x1 * width / 1000))),
        max(0, min(height - 1, round(y1 * height / 1000))),
        max(1, min(width, round(x2 * width / 1000))),
        max(1, min(height, round(y2 * height / 1000))),
    )
    if box[2] <= box[0] or box[3] <= box[1]:
        raise ValueError(f"Target box không hợp lệ: {box}")
    return box


def create_box_mask(image_size, box):
    """Create a binary rectangular inpainting mask."""
    mask = Image.new("L", image_size, 0)
    ImageDraw.Draw(mask).rectangle(box, fill=255)
    return mask


In [ ]:
if RUN_POOL_SELECTION:
    background_pool = []
    for pool_index, selection in enumerate(qwen_selections, start=1):
        selected_item = background_candidates[selection["image_index"]]
        selected_background = download_open_image(
            selected_item["source_url"], CONFIG["max_background_side"]
        )
        target_box = normalized_box_to_pixels(selection["box_2d"], selected_background.size)
        background_id = f"bg{pool_index:04d}"
        background_path = BACKGROUND_DIR / f"{background_id}.jpg"
        selected_background.save(background_path, quality=95)
        background_pool.append({
            "background_id": background_id,
            "prompt_id": selection["prompt_id"],
            "source_index": selected_item["source_index"],
            "source_url": selected_item["source_url"],
            "qwen_box_2d": selection["box_2d"],
            "scene_type": selection["scene_type"],
            "matched_elements": selection["matched_elements"],
            "match_reason": selection["match_reason"],
            "target_box": list(target_box),
            "background_path": str(background_path.relative_to(PROJECT_DIR)),
        })
    experiment = {
        "name": CONFIG["experiment_name"],
        "created_at": datetime.now().isoformat(),
        "config": CONFIG,
        "prompts": PROMPTS,
        "background_pool": background_pool,
        "qwen_selection_log": qwen_selection_log,
        "adaptive_decisions": [],
        "best_outputs": [],
    }
    EXPERIMENT_PATH.write_text(
        json.dumps(experiment, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
else:
    if not EXPERIMENT_PATH.exists():
        raise FileNotFoundError(
            "experiment.json chưa tồn tại; hãy đặt RUN_POOL_SELECTION=True"
        )
    experiment = json.loads(EXPERIMENT_PATH.read_text(encoding="utf-8"))
    if experiment.get("prompts") != PROMPTS:
        raise ValueError(
            "PROMPTS đã thay đổi; hãy đặt RUN_POOL_SELECTION=True để tạo lại pool"
        )
    background_pool = experiment["background_pool"]

pool_counts = {item["prompt_id"]: 0 for item in PROMPTS}
for item in background_pool:
    if item["prompt_id"] not in pool_counts:
        raise ValueError(f"prompt_id không hợp lệ trong Background Pool: {item['prompt_id']}")
    pool_counts[item["prompt_id"]] += 1
if any(
    not CONFIG["min_background_count"] <= count <= CONFIG["target_background_count"]
    for count in pool_counts.values()
):
    raise ValueError(
        "Background Pool không đạt số lượng cấu hình cho mỗi prompt: "
        f"{pool_counts}"
    )

for item in background_pool:
    preview = Image.open(PROJECT_DIR / item["background_path"]).convert("RGB")
    ImageDraw.Draw(preview).rectangle(
        tuple(item["target_box"]),
        outline="red",
        width=max(3, preview.width // 300),
    )
    print(
        item["background_id"], item["prompt_id"], item["scene_type"],
        item["matched_elements"], "—", item["match_reason"],
    )
    display(preview)

## 7. SDXL Inpainting

Cell này quản lý generation size, lazy model loading, mask preparation và seeded inpainting. Pixels ngoài hard mask được composite từ background gốc.

In [ ]:
def generation_size(image_size):
    """Fit an image to SDXL limits using dimensions divisible by eight."""
    width, height = image_size
    scale = min(
        CONFIG["max_generation_width"] / width,
        CONFIG["max_generation_height"] / height,
        1.0,
    )
    return (
        max(8, int(width * scale) // 8 * 8),
        max(8, int(height * scale) // 8 * 8),
    )


def ensure_sdxl_pipeline():
    """Lazy-load the SDXL inpainting pipeline once per runtime."""
    global sdxl_pipe
    if sdxl_pipe is not None:
        return
    assert torch.cuda.is_available(), "SDXL generation cần GPU"
    sdxl_pipe = AutoPipelineForInpainting.from_pretrained(
        CONFIG["sdxl_model_id"],
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
        token=HF_TOKEN,
    )
    sdxl_pipe.enable_model_cpu_offload()
    sdxl_pipe.set_progress_bar_config(disable=False)


def prepare_inpaint_inputs(background_image, target_mask):
    """Resize the background and prepare hard and feathered masks."""
    size = generation_size(background_image.size)
    background = background_image.resize(size, Image.Resampling.LANCZOS)
    hard_mask = target_mask.resize(size, Image.Resampling.NEAREST)
    soft_mask = hard_mask.filter(ImageFilter.GaussianBlur(CONFIG["mask_blur_radius"]))
    return background, hard_mask, soft_mask


def generate_candidate(seed, prompt, selected_background, target_mask):
    """Generate one seeded candidate and preserve pixels outside the hard mask."""
    ensure_sdxl_pipeline()
    background, hard_mask, soft_mask = prepare_inpaint_inputs(
        selected_background, target_mask
    )
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    generated = sdxl_pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=background,
        mask_image=soft_mask,
        generator=generator,
        num_inference_steps=CONFIG["num_inference_steps"],
        guidance_scale=CONFIG["guidance_scale"],
        strength=CONFIG["strength"],
        width=background.width,
        height=background.height,
    ).images[0]
    return Image.composite(generated, background, hard_mask), background, hard_mask

## 8. Automatic evaluator

Các cells tách model lifecycle, box geometry, model inference và candidate aggregation. Evaluator ghi năm signals phục vụ adaptive search:

- person presence;
- placement inside the target region;
- scale;
- prompt alignment;
- background preservation ngoài target box.

Orientation, activity và interaction được phản ánh qua prompt alignment trong v1.


In [ ]:
def ensure_evaluator_models():
    """Lazy-load instance segmentation and CLIP evaluator models."""
    global instance_processor, instance_model, clip_processor, clip_model
    models = (instance_processor, instance_model, clip_processor, clip_model)
    if all(model is not None for model in models):
        return

    assert torch.cuda.is_available(), "Automatic evaluator cần GPU"
    instance_processor = AutoImageProcessor.from_pretrained(
        CONFIG["instance_model_id"],
        token=HF_TOKEN,
    )
    instance_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        CONFIG["instance_model_id"],
        token=HF_TOKEN,
    ).to("cuda").eval()

    clip_processor = CLIPProcessor.from_pretrained(
        CONFIG["clip_model_id"],
        token=HF_TOKEN,
    )
    clip_model = CLIPModel.from_pretrained(
        CONFIG["clip_model_id"],
        token=HF_TOKEN,
    ).to("cuda").eval()


In [ ]:
def bbox_from_mask(mask):
    """Return the tight pixel box of a binary mask."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()) + 1,
        int(ys.max()) + 1,
    )


def resize_box(box, old_size, new_size):
    """Scale a box between image coordinate systems."""
    sx = new_size[0] / old_size[0]
    sy = new_size[1] / old_size[1]
    return (
        float(box[0] * sx),
        float(box[1] * sy),
        float(box[2] * sx),
        float(box[3] * sy),
    )


def box_iou(a, b):
    """Compute intersection-over-union for two boxes."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return intersection / max(1e-6, area_a + area_b - intersection)


def center_score(person_box, target_box):
    """Score the distance between person and target-box centers."""
    px = (person_box[0] + person_box[2]) / 2
    py = (person_box[1] + person_box[3]) / 2
    tx = (target_box[0] + target_box[2]) / 2
    ty = (target_box[1] + target_box[3]) / 2
    diagonal = math.hypot(
        target_box[2] - target_box[0],
        target_box[3] - target_box[1],
    )
    distance = math.hypot(px - tx, py - ty)
    return float(np.exp(-2.0 * distance / max(1.0, diagonal)))


def background_integrity_score(image, background, target_box):
    """Measure pixel preservation outside the target box."""
    reference = background.resize(image.size, Image.Resampling.LANCZOS)
    output_array = np.asarray(image, dtype=np.float32)
    reference_array = np.asarray(reference, dtype=np.float32)
    outside_mask = np.ones((image.height, image.width), dtype=bool)

    x1, y1, x2, y2 = [int(round(value)) for value in target_box]
    x1, x2 = np.clip([x1, x2], 0, image.width)
    y1, y2 = np.clip([y1, y2], 0, image.height)
    outside_mask[y1:y2, x1:x2] = False
    if not outside_mask.any():
        return 0.0

    absolute_error = np.abs(output_array - reference_array).mean(axis=2)
    score = 1.0 - float(absolute_error[outside_mask].mean() / 255.0)
    return float(np.clip(score, 0.0, 1.0))


In [ ]:
@torch.inference_mode()
def segment_people(image):
    """Detect person instances and return their bounding boxes."""
    ensure_evaluator_models()
    inputs = instance_processor(
        images=image,
        return_tensors="pt",
    ).to("cuda")
    outputs = instance_model(**inputs)
    result = instance_processor.post_process_instance_segmentation(
        outputs,
        threshold=CONFIG["person_detection_threshold"],
        target_sizes=[(image.height, image.width)],
        return_binary_maps=True,
    )[0]

    maps = result.get("segmentation")
    if maps is None:
        return []

    maps = maps.detach().cpu().numpy()
    if maps.ndim == 2:
        maps = maps[None]

    people = []
    for index, info in enumerate(result["segments_info"]):
        label = instance_model.config.id2label.get(
            info["label_id"],
            "",
        ).lower()
        if label != "person" or index >= len(maps):
            continue

        mask = maps[index].astype(bool)
        box = bbox_from_mask(mask)
        if box is not None:
            people.append({"box": box})

    return people


@torch.inference_mode()
def clip_alignment(image, prompt):
    """Compute normalized CLIP similarity for an image-prompt pair."""
    ensure_evaluator_models()
    inputs = clip_processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to("cuda")

    image_features = F.normalize(
        pooled_feature_tensor(
            clip_model.get_image_features(
                pixel_values=inputs.pixel_values,
            )
        ),
        dim=-1,
    )
    text_features = F.normalize(
        pooled_feature_tensor(
            clip_model.get_text_features(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
            )
        ),
        dim=-1,
    )
    cosine = float((image_features * text_features).sum().item())
    return float(np.clip((cosine + 1.0) / 2.0, 0.0, 1.0))


In [ ]:
def evaluate_candidate(record, background, target_box):
    """Evaluate one generated candidate with the five ALTO signals."""
    metric_names = [
        "placement", "scale", "prompt_alignment",
        "background_preservation",
    ]

    if record["status"] != "success":
        return {
            **record,
            "person_detected": False,
            **{name: 0.0 for name in metric_names},
        }

    image = Image.open(record["output_path"]).convert("RGB")
    resized_target = resize_box(
        target_box,
        background.size,
        image.size,
    )
    people = segment_people(image)
    selected = (
        max(
            people,
            key=lambda item: center_score(
                item["box"],
                resized_target,
            ),
        )
        if people
        else None
    )

    if selected is None:
        placement = 0.0
        scale = 0.0
    else:
        person_box = selected["box"]
        placement = (
            0.55 * center_score(person_box, resized_target)
            + 0.45 * box_iou(person_box, resized_target)
        )
        person_height = max(1.0, person_box[3] - person_box[1])
        target_height = max(1.0, resized_target[3] - resized_target[1])
        scale = float(
            np.exp(
                -abs(math.log(person_height / target_height)) / 0.45
            )
        )
    return {
        **record,
        "person_detected": selected is not None,
        "placement": float(placement),
        "scale": float(scale),
        "prompt_alignment": clip_alignment(
            image,
            record["prompt"],
        ),
        "background_preservation": background_integrity_score(
            image, background, resized_target
        ),
    }

## 9. Adaptive budget policy

Các cells lần lượt chuẩn hóa thresholds/acceptance, rank candidates, rồi quyết định accept, stop, expand lên 8 hoặc expand lên 16.


In [ ]:
METRIC_THRESHOLDS = {
    "placement": CONFIG["min_placement"],
    "scale": CONFIG["min_scale"],
    "prompt_alignment": CONFIG["min_prompt_alignment"],
    "background_preservation": CONFIG["min_background_preservation"],
}
METRIC_COLUMNS = list(METRIC_THRESHOLDS)


def as_bool(value):
    """Normalize CSV-compatible boolean values."""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


def accepted(row):
    """Return whether a candidate passes every acceptance constraint."""
    return bool(
        as_bool(row["person_detected"])
        and all(
            float(row[name]) >= threshold
            for name, threshold in METRIC_THRESHOLDS.items()
        )
    )


In [ ]:
def select_best(frame):
    """Rank successful candidates with deterministic tie-breaking."""
    successful = frame[frame["status"] == "success"].copy()
    if successful.empty:
        return None

    successful[METRIC_COLUMNS] = successful[METRIC_COLUMNS].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0.0)
    successful["seed"] = pd.to_numeric(
        successful["seed"], errors="coerce"
    ).fillna(float("inf"))
    successful["_accepted"] = successful.apply(
        accepted,
        axis=1,
    )
    successful["_person"] = successful["person_detected"].map(as_bool)
    successful["_hard_passes"] = sum(
        (successful[name] >= threshold).astype(int)
        for name, threshold in METRIC_THRESHOLDS.items()
    )
    successful["_quality"] = successful[METRIC_COLUMNS].mean(axis=1)

    return successful.sort_values(
        [
            "_accepted",
            "_person",
            "_hard_passes",
            "_quality",
            "seed",
        ],
        ascending=[False, False, False, False, True],
    ).iloc[0]


def is_promising(frame):
    """Check whether the best detected person is near all thresholds."""
    successful_people = frame[
        (frame["status"] == "success")
        & frame["person_detected"].map(as_bool)
    ].copy()
    if successful_people.empty:
        return False

    best = select_best(successful_people)
    if best is None:
        return False

    margin = CONFIG["promising_margin"]
    return all(
        float(best[name]) >= threshold - margin
        for name, threshold in METRIC_THRESHOLDS.items()
    )


In [ ]:
def adaptive_decision(evaluation_frame):
    """Choose accept, stop or the next candidate budget."""
    observed = len(evaluation_frame)
    best = select_best(evaluation_frame)

    if best is not None and accepted(best):
        return "accept", "all_thresholds_passed"

    if evaluation_frame["status"].ne("success").all():
        return "stop", "all_generations_failed"

    # Four seeds are an exploration round. Do not reject a task this early
    # because generation quality is highly seed-sensitive.
    if observed < 8:
        return "expand_to_8", "minimum_observation_budget"

    has_person = evaluation_frame["person_detected"].map(as_bool).any()
    if not has_person:
        return "stop", "no_person_after_8"

    if observed < 16:
        if is_promising(evaluation_frame):
            return "expand_to_16", "near_threshold_after_8"
        return "stop", "not_promising_after_8"

    return "stop", "maximum_budget_reached"

## 10. Run protocol

Các cells lần lượt load background context, generate từng round, persist protocol state, reload/re-evaluate kết quả cũ và dispatch theo execution flags. `full_fixed` sinh đủ 16 seeds; `adaptive` dừng theo policy.


In [ ]:
prompt_by_id = {
    item["prompt_id"]: item["generation_prompt"]
    for item in PROMPTS
}
pool_by_id = {
    item["background_id"]: item
    for item in background_pool
}


def load_background_context(pool_item):
    """Load one background and derive its reusable target mask."""
    background_id = pool_item["background_id"]
    selected_background = Image.open(
        PROJECT_DIR / pool_item["background_path"]
    ).convert("RGB")
    target_box = tuple(pool_item["target_box"])
    target_mask = create_box_mask(
        selected_background.size,
        target_box,
    )
    target_mask.save(MASK_DIR / f"{background_id}.png")
    return {
        "background_id": background_id,
        "image": selected_background,
        "target_box": target_box,
        "target_mask": target_mask,
    }


In [ ]:
def generate_round(seeds, round_index, prompt_id, prompt, context):
    """Generate and record all seeds in one adaptive round."""
    records = []
    background_id = context["background_id"]
    output_dir = OUTPUT_DIR / "images" / background_id
    output_dir.mkdir(parents=True, exist_ok=True)

    for seed in seeds:
        candidate_id = f"{background_id}_s{seed}"
        output_path = output_dir / f"{candidate_id}.png"
        record = {
            "candidate_id": candidate_id,
            "background_id": background_id,
            "prompt_id": prompt_id,
            "prompt": prompt,
            "seed": int(seed),
            "round": int(round_index),
            "output_path": str(output_path),
            "status": "started",
            "elapsed_seconds": None,
            "error": "",
        }

        try:
            start = time.time()
            image, _, _ = generate_candidate(
                seed, prompt, context["image"], context["target_mask"]
            )
            image.save(output_path)
            record["status"] = "success"
            record["elapsed_seconds"] = time.time() - start
            print(
                candidate_id,
                f"{record['elapsed_seconds']:.1f}s",
            )
        except Exception as exc:
            record["status"] = "failed"
            record["error"] = repr(exc)
            print("FAILED", candidate_id, exc)

        records.append(record)

    return records


def terminal_budget(decision_frame):
    """Return the effective budget at the first terminal decision."""
    terminal = decision_frame[
        decision_frame["decision"].isin(["accept", "stop"])
    ]
    if terminal.empty:
        return int(decision_frame["observed_candidates"].max())
    return int(terminal.iloc[0]["observed_candidates"])


In [ ]:
def write_run_outputs(run_records, evaluation_records):
    """Persist cumulative run and evaluation records."""
    runs = pd.DataFrame(run_records)
    evaluations = pd.DataFrame(evaluation_records)
    runs.to_csv(RUNS_PATH, index=False, encoding="utf-8-sig")
    evaluations.to_csv(EVALUATIONS_PATH, index=False, encoding="utf-8-sig")
    return runs, evaluations


def run_generation_protocol():
    """Execute generation, evaluation and adaptive decisions for the pool."""
    if not RUN_EVALUATION:
        raise ValueError("Generation protocol cần RUN_EVALUATION=True")

    run_records, evaluation_records, decision_records = [], [], []
    for pool_item in background_pool:
        prompt_id = pool_item["prompt_id"]
        prompt = prompt_by_id[prompt_id]
        context = load_background_context(pool_item)
        current_evaluations = []
        next_seed = CONFIG["seed_start"]

        for round_index, round_count in enumerate(CONFIG["round_seed_counts"]):
            seeds = list(range(next_seed, next_seed + round_count))
            next_seed += round_count
            round_runs = generate_round(
                seeds, round_index, prompt_id, prompt, context
            )
            round_evaluations = [
                evaluate_candidate(
                    record, context["image"], context["target_box"]
                )
                for record in round_runs
            ]
            run_records.extend(round_runs)
            evaluation_records.extend(round_evaluations)
            current_evaluations.extend(round_evaluations)
            runs, evaluations = write_run_outputs(
                run_records, evaluation_records
            )

            current_frame = pd.DataFrame(current_evaluations)
            decision, reason = adaptive_decision(current_frame)
            best = select_best(current_frame)
            decision_records.append({
                "background_id": context["background_id"],
                "prompt_id": prompt_id,
                "round": round_index,
                "observed_candidates": len(current_frame),
                "decision": decision,
                "reason": reason,
                "best_candidate_id": None if best is None else best["candidate_id"],
                "simulated": CONFIG["generation_protocol"] == "full_fixed",
            })
            print(
                f"{context['background_id']}, round {round_index}: "
                f"{decision} ({reason})"
            )
            if (
                CONFIG["generation_protocol"] == "adaptive"
                and decision in {"accept", "stop"}
            ):
                break

    experiment["adaptive_decisions"] = decision_records
    experiment["config"] = CONFIG
    EXPERIMENT_PATH.write_text(
        json.dumps(experiment, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return runs, evaluations

In [ ]:
def load_existing_results():
    """Load prior runs and optionally recompute their evaluations."""
    if not RUNS_PATH.exists():
        raise FileNotFoundError(
            "runs.csv chưa tồn tại; hãy đặt RUN_GENERATION=True"
        )
    runs = pd.read_csv(RUNS_PATH, encoding="utf-8-sig")

    if not RUN_EVALUATION:
        if not EVALUATIONS_PATH.exists():
            raise FileNotFoundError(
                "evaluations.csv chưa tồn tại; hãy đặt RUN_EVALUATION=True"
            )
        return runs, pd.read_csv(EVALUATIONS_PATH, encoding="utf-8-sig")

    unknown_backgrounds = set(runs["background_id"]) - set(pool_by_id)
    if unknown_backgrounds:
        raise ValueError(
            f"runs.csv chứa background_id không thuộc pool: "
            f"{sorted(unknown_backgrounds)}"
        )

    evaluation_records = []
    for background_id, group in runs.groupby("background_id", sort=False):
        context = load_background_context(pool_by_id[background_id])
        evaluation_records.extend(
            evaluate_candidate(
                row, context["image"], context["target_box"]
            )
            for row in group.to_dict("records")
        )
    evaluations = pd.DataFrame(evaluation_records)
    evaluations.to_csv(EVALUATIONS_PATH, index=False, encoding="utf-8-sig")
    return runs, evaluations

In [ ]:
if RUN_GENERATION:
    runs_df, evaluations_df = run_generation_protocol()
else:
    runs_df, evaluations_df = load_existing_results()

required_metric_columns = {
    "person_detected", *METRIC_COLUMNS,
}
missing_columns = required_metric_columns - set(evaluations_df.columns)
if missing_columns:
    raise ValueError(
        f"evaluations.csv thiếu columns {sorted(missing_columns)}; "
        "hãy đặt RUN_EVALUATION=True để tạo lại."
    )

display(evaluations_df)

## 11. Best outputs and experiment manifest

Cell selection xác định adaptive best và oracle Best-of-16 cho từng background. Cell summary tính effective budget, compute saving ratio và cập nhật `experiment.json`.


In [ ]:
adaptive_best_outputs = []
oracle_best_outputs = []
effective_budgets = []

decision_frame = pd.DataFrame(
    experiment.get("adaptive_decisions", [])
)

for current_background_id, frame in evaluations_df.groupby(
    "background_id",
    sort=False,
):
    ordered = frame.sort_values(
        ["round", "seed"],
        ascending=[True, True],
    ).reset_index(drop=True)

    if decision_frame.empty or "background_id" not in decision_frame:
        background_decisions = pd.DataFrame()
    else:
        background_decisions = decision_frame[
            decision_frame["background_id"] == current_background_id
        ].sort_values("observed_candidates")

    if background_decisions.empty:
        effective_budget = len(ordered)
    else:
        effective_budget = terminal_budget(background_decisions)
    effective_budgets.append(effective_budget)

    adaptive_frame = ordered.head(effective_budget)
    adaptive_best = select_best(adaptive_frame)
    oracle_best = select_best(ordered)

    if adaptive_best is not None:
        adaptive_best_outputs.append({
            "background_id": current_background_id,
            "prompt_id": adaptive_best["prompt_id"],
            "candidate_id": adaptive_best["candidate_id"],
            "output_path": adaptive_best["output_path"],
            "accepted": accepted(adaptive_best),
            "effective_budget": int(effective_budget),
            "quality_score": float(
                adaptive_best[METRIC_COLUMNS].mean()
            ),
            **{
                name: float(adaptive_best[name])
                for name in METRIC_COLUMNS
            },
        })
        display(
            current_background_id,
            Image.open(adaptive_best["output_path"]),
        )

    if oracle_best is not None:
        oracle_best_outputs.append({
            "background_id": current_background_id,
            "prompt_id": oracle_best["prompt_id"],
            "candidate_id": oracle_best["candidate_id"],
            "output_path": oracle_best["output_path"],
            "accepted": accepted(oracle_best),
            "oracle_budget": int(len(ordered)),
            "quality_score": float(
                oracle_best[METRIC_COLUMNS].mean()
            ),
            **{
                name: float(oracle_best[name])
                for name in METRIC_COLUMNS
            },
        })


In [ ]:
if not adaptive_best_outputs:
    raise RuntimeError("Không có output thành công")

average_budget = float(
    np.mean(effective_budgets)
)
maximum_budget = sum(CONFIG["round_seed_counts"])
compute_saving = float(
    1.0 - average_budget / maximum_budget
)

experiment["completed_at"] = datetime.now().isoformat()
experiment["runs_path"] = str(RUNS_PATH)
experiment["evaluations_path"] = str(EVALUATIONS_PATH)
experiment["generated_candidates"] = int(len(runs_df))
experiment["best_outputs"] = adaptive_best_outputs
experiment["oracle_best_outputs"] = oracle_best_outputs
experiment["adaptive_summary"] = {
    "average_effective_budget": average_budget,
    "maximum_budget": maximum_budget,
    "compute_saving_ratio": compute_saving,
}

EXPERIMENT_PATH.write_text(
    json.dumps(
        experiment,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(json.dumps(
    {
        "adaptive_summary": experiment["adaptive_summary"],
        "best_outputs": adaptive_best_outputs,
    },
    indent=2,
    ensure_ascii=False,
))